## Execute Semantic Similarity Queries in Azure Cosmos DB

### Installing Libraries and Utilities

In [ ]:
%pip install azure-cosmos==4.16.0 azure-identity python-dotenv openai==2.38.0

### Setting up your Environment

In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

# fetching the cosmosdb configuration from environment variables
cosmosdb_endpoint = os.getenv("COSMOSDB_ENDPOINT")
cosmosdb_key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME") + "Vector"

# fetching the azure openai configuration from environment variables
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")
chat_completions_model_name = os.getenv("CHAT_COMPLETIONS_MODEL_NAME")

### Creating the Cosmos DB Client

In [2]:
from azure.cosmos import CosmosClient
from azure.cosmos import PartitionKey

client = CosmosClient(cosmosdb_endpoint, cosmosdb_key)

### Navigate the Resource Hierarchy

In [3]:
database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

### Create the Azure OpenAI Client

In [4]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_key,
    api_version="2024-02-15-preview",
    azure_endpoint=azure_openai_endpoint
)

### Creating the Embedding Generator Helper Function

In [5]:
def generate_embeddings(client, text):
    
    response = client.embeddings.create(
        input=text,
        model = embedding_model_name
    )
    
    embeddings=response.model_dump()
    return embeddings['data'][0]['embedding']
    

### Execute Your First Vector Search Query

In [6]:
query = """
SELECT TOP 5
    c.id,
    c.name,
    c.category,
    c.content,
    VectorDistance(
        c.vector,
        @queryVector
    ) AS SimilarityScore
FROM c
ORDER BY VectorDistance(
    c.vector,
    @queryVector
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "high protein food")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print("name: {}, category: {}, similarity score: {}".format(item['name'], item['category'], item['SimilarityScore']))
    print("-----------")
    print("content: {}".format(item['content']))
    print("====================================")
    print("\n\n")

name: Premium Family Feast with Cookies, category: Sandwiches, similarity score: 0.3148771095768617
-----------
content: Premium Family Feast with Cookies belongs to category Sandwiches. A meal to feed the whole family. Includes 4 Half Sandwiches, 1 Whole Salad, 1 Quart of Soup, 1 Whole French Baguette, and 4 Chocolate Chippers. Serves 4-6. Price is 45.79 USD.



name: Premium Family Feast, category: Sandwiches, similarity score: 0.3096029235629097
-----------
content: Premium Family Feast belongs to category Sandwiches. A meal to feed the whole family. Includes 4 Half Sandwiches, 1 Whole Salad, 1 Quart of Soup, and 1 Whole French Baguette. Serves 4-6. Price is 41.09 USD.



name: Go Getter, category: Smoothies, similarity score: 0.30563415815856804
-----------
content: Go Getter belongs to category Smoothies. banana, peanut butter, raw cacao, cinnamon, hemp seeds, walnuts, almond milk, chocolate vegan protein Price is 10.8 USD.



name: Toasted Frontega Chicken, category: Sandwiches, 

### Apply a Similarity Threshold to Filter Results

In [8]:
query = """
SELECT TOP 5
    c.name,
    VectorDistance(
        c.vector,
        @queryVector
    ) AS SimilarityScore
FROM c
WHERE VectorDistance(
    c.vector,
    @queryVector
) > 0.3
ORDER BY VectorDistance(
    c.vector,
    @queryVector
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "high protein food")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print(
        item["name"],
        item["SimilarityScore"]
    )

Premium Family Feast with Cookies 0.3148771095768617
Premium Family Feast 0.3096029235629097
Go Getter 0.30563415815856804


### Implement a Brute-Force Similarity Search

In [9]:
query = """
SELECT TOP 5
    c.name,
    VectorDistance(
        c.vector,
        @queryVector,
        true
    ) AS SimilarityScore
FROM c
ORDER BY VectorDistance(
    c.vector,
    @queryVector,
    true
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "high protein food")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print(
        item["name"],
        item["SimilarityScore"]
    )

Premium Family Feast with Cookies 0.3148771095768617
Premium Family Feast 0.3096029235629097
Go Getter 0.30563415815856804
Toasted Frontega Chicken 0.2870117517737857
Green Boi 0.286715039806882


### Perform Vector Search with Metadata Filtering

In [10]:
query = """
SELECT TOP 5
    c.id,
    c.name,
    c.category,
    c.dietaryTags,
    VectorDistance(
        c.vector,
        @queryVector
    ) AS SimilarityScore
FROM c
WHERE ARRAY_CONTAINS(
    c.dietaryTags,
    "Vegetarian"
)
ORDER BY VectorDistance(
    c.vector,
    @queryVector
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "high protein food")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print("name: {}, category: {}, dietary tags: {}, similarity score: {}".format(
        item['name'],
        item['category'],
        item['dietaryTags'],
        item['SimilarityScore']
    ))

name: Go Getter, category: Smoothies, dietary tags: ['Low Carb', 'Vegetarian', 'Vegan'], similarity score: 0.30563415815856804
name: Toasted Frontega Chicken, category: Sandwiches, dietary tags: ['Low Carb', 'Vegetarian'], similarity score: 0.2870117517737857
name: Green Boi, category: Smoothies, dietary tags: ['Vegan', 'Vegetarian', 'Low Carb'], similarity score: 0.286715039806882
name: NEW The Spicy Take Chicken Sandwich, category: Sandwiches, dietary tags: ['Vegetarian', 'Gluten-Free'], similarity score: 0.2789281299384943
name: Classic Grilled Cheese Sandwich, category: Sandwiches, dietary tags: ['Vegan', 'Vegetarian'], similarity score: 0.2600453759895574


### Optimize Search with Partitioning 

In [11]:
query = """
SELECT TOP 5
    c.id,
    c.name,
    c.category,
    c.dietaryTags,
    VectorDistance(
        c.vector,
        @queryVector
    ) AS SimilarityScore
FROM c
WHERE ARRAY_CONTAINS(
    c.dietaryTags,
    "Vegetarian"
)
ORDER BY VectorDistance(
    c.vector,
    @queryVector
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "high protein food")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        }
    ],
    partition_key="Smoothies",
)

for item in results:
    print("name: {}, category: {}, dietary tags: {}, similarity score: {}".format(
        item['name'],
        item['category'],
        item['dietaryTags'],
        item['SimilarityScore']
    ))

name: Go Getter, category: Smoothies, dietary tags: ['Low Carb', 'Vegetarian', 'Vegan'], similarity score: 0.30563415815856804
name: Green Boi, category: Smoothies, dietary tags: ['Vegan', 'Vegetarian', 'Low Carb'], similarity score: 0.286715039806882
name: Jimmy Jam Smoothie, category: Smoothies, dietary tags: ['Vegetarian', 'Gluten-Free'], similarity score: 0.23335496419835952
name: Early Birds Get the Worm Smoothie, category: Smoothies, dietary tags: ['Gluten-Free', 'Vegetarian'], similarity score: 0.22377304657490169
name: Aw Shuckie Shuckie Now Smoothie, category: Smoothies, dietary tags: ['Vegetarian', 'Gluten-Free', 'Low Carb'], similarity score: 0.2040358120336984
